# Etapa 7b — Reimplementar el loop de la Etapa 6 con LangGraph

**Proyecto Retrieval & Memory Lab · corpus jurídico argentino**

Se compara la implementación propia de **Etapa 6** (`memory` y `memory_filter`) contra la **orquestación LangGraph** de *las mismas funciones congeladas*. Las sondas provienen de `eval/scenarios.jsonl`, no se modifican. Como la Etapa 6 usa el loop de la Etapa 5, se reimplementa el camino completo: escritura de sesión → almacenamiento entre sesiones → recuperación BM25 → lectura de memoria → respuesta con Qwen3.

**Qué cambia:** `StateGraph` organiza los nodos, `InMemorySaver` persiste estados por hilo y `InMemoryStore` conserva registros entre hilos. **Qué no cambia:** el extractor, los esquemas de respuesta, el criterio ADD/UPDATE/NOOP/INVALIDATE, el lector de memorias, los filtros, BM25, la construcción del prompt, los presupuestos, el tokenizer y la función de evaluación. De este modo evaluamos la orquestación/persistencia, **no** una nueva estrategia de memoria. El framework *no decide automáticamente* cuándo resumir ni qué hechos guardar.

**Ejecución:** usar Colab con GPU T4. Primero probar opcionalmente en `dev` / smoke; luego poner `SMOKE_TEST=False`, `SPLIT='test'` en la celda de configuración. El notebook necesita los artefactos de Etapas 2/3 y la implementación de Etapa 6 del repositorio.

**Importante:** los resultados de LangGraph se obtienen **cuando corras las celdas**. Este notebook incluye código y validaciones, no inventa resultados de test.

In [3]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

artifacts -> /content/drive/MyDrive/projecto_rag/artifacts
Colab bootstrap OK. Working directory: /content/projecto_rag


In [2]:
from pathlib import Path
import os
import shutil
from datetime import datetime

repo = Path("/content/projecto_rag")
local = repo / "artifacts"
drive = Path("/content/drive/MyDrive/projecto_rag/artifacts")
drive.mkdir(parents=True, exist_ok=True)

if local.is_symlink():
    print("Ya existe el enlace:", local.resolve())

else:
    if local.is_dir():
        archivos = [p for p in local.rglob("*") if p.is_file()]

        if not archivos or all(p.name == ".gitkeep" for p in archivos):
            shutil.rmtree(local)
        else:
            backup = drive.parent / (
                "artifacts_respaldo_" +
                datetime.now().strftime("%Y%m%d_%H%M%S")
            )
            shutil.move(str(local), str(backup))
            print("Archivos locales respaldados en:", backup)

    os.symlink(str(drive), str(local))

print("artifacts ->", local.resolve())
print("¿Es enlace simbólico?", local.is_symlink())

Archivos locales respaldados en: /content/drive/MyDrive/projecto_rag/artifacts_respaldo_20261101_225734
artifacts -> /content/drive/MyDrive/projecto_rag/artifacts
¿Es enlace simbólico? True


## Dependencias del framework

Copiá la línea de `langgraph` desde `requirements-extras.txt` a `requirements-group.txt` (sin borrar las dependencias existentes). La siguiente celda instalará la línea del archivo de extras si el framework todavía no está instalado; registra la versión real para reproducibilidad.

Se usa el endpoint nativo del proyecto `llm.structured` (Ollama) y **no** la API OpenAI-compatible `/v1`, para mantener `num_ctx=8192` y `think=False` en cada llamada.

In [4]:
import importlib.util, subprocess, sys
from pathlib import Path
if importlib.util.find_spec('langgraph') is None:
    extras = Path('requirements-extras.txt')
    if not extras.exists():
        raise FileNotFoundError('Falta requirements-extras.txt: copiá la dependencia LangGraph a requirements-group.txt e instalala, según la consigna.')
    extra_lines = [line.strip() for line in extras.read_text(encoding='utf-8').splitlines()
                   if line.strip().lower().startswith('langgraph') and not line.lstrip().startswith('#')]
    if not extra_lines:
        raise RuntimeError('No encontré la línea LangGraph en requirements-extras.txt; revisá requirements-group.txt.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', extra_lines[0]], check=True)
import importlib.metadata
LANGGRAPH_VERSION = importlib.metadata.version('langgraph')
print('LangGraph instalado:', LANGGRAPH_VERSION)


LangGraph instalado: 1.2.18


## Preparación idéntica a la Etapa 6

Las celdas siguientes fueron copiadas de `06_memory.ipynb` para mantener las mismas guardas de corpus, índice, modelo y ventana. **No ejecutar el loop de evaluación original de ese notebook**: lo sustituimos por un grafo.

In [5]:
import environment  # primero: mismas guardas del proyecto (JAX, caché de HF)
from dataclasses import asdict
from pathlib import Path
import ast
import csv
import hashlib
import json
import shutil
import sys
import time
import types
import warnings

SMOKE_TEST = False
SPLIT = 'test'               # 'test' solo cuando las decisiones estén fijadas
PROFILE = 'smoke' if SMOKE_TEST else 'full'
MODEL = 'qwen3:1.7b' if SMOKE_TEST else 'qwen3:8b'
RUN_SWEEP = not SMOKE_TEST  # brazos replay_50 / replay_25 (dónde deja de entrar la reproducción)
EMBEDDER = 'bge-m3'         # 'bge-m3' (el de la Etapa 3) o 'lexical' (respaldo declarado, sin GPU)
STAGE3_INDEX_DIR = None     # opcional: Path('/content/drive/MyDrive/.../index/stage3_xxxx') si el índice está en otro lado


ART = environment.artifacts_dir(SMOKE_TEST)          # resultados de esta corrida
ART_FULL = environment.artifacts_dir(False)          # corpus e índice: siempre los completos
DOC_PATH = ART_FULL / 'documents.jsonl'
EVAL_DIR = Path('eval')
DRIVE_ART = Path('/content/drive/MyDrive/projecto_rag/artifacts')

assert SPLIT in ('dev', 'test')
assert not (SPLIT == 'test' and SMOKE_TEST), 'Nunca evaluar test con smoke.'
if environment.on_colab() and not environment.ARTIFACTS.is_symlink():
    print('AVISO: artifacts/ no está enlazado a Drive. Se leen el corpus y el índice desde Drive si faltan, '
          'y al final los resultados se copian a Drive.')
environment.summary()

ENVIRONMENT
os             : Linux 6.6.159+
python         : 3.13.16
interpreter    : /usr/bin/python3
colab          : True
logical_cores  : 2
torch          : 2.11.0+cu130
cuda_available : True
gpu            : Tesla T4
gpu_memory_gb  : 15.64
ram_total_gb   : 13.6
ram_free_gb    : 10.4
HF_HOME        : /content/drive/MyDrive/projecto_rag/hf_cache
jax_guard      : True
ca_bundle      : not present (normal)
artifacts      : -> /content/drive/MyDrive/projecto_rag/artifacts (Drive)


In [6]:
"""Etapa 6: memoria entre sesiones.

El notebook notebooks/06_memory incluye una copia de este módulo (como las Etapas 3 y 4),
así puede abrirse solo en Colab. Reusa sin modificarlo `grounded_stage4.py` (el ensamblador,
el schema de respuesta y la contención de la clave), igual que el loop de la Etapa 5.

Qué contiene:
  - Esquemas: CandidateFact / Extraction (camino de escritura), WriteDecision, MemoryRecord.
    En todos, el razonamiento se declara primero y su nombre también queda primero
    alfabéticamente ("analysis" < ...).
  - MemoryStore: el almacén. Invalidá, no borres: un UPDATE cierra el registro viejo
    (valid_to) y agrega uno nuevo; nunca se elimina nada. Todo queda en un log de operaciones.
  - write_session: extraer hechos con un schema -> recuperar los recuerdos parecidos ->
    decidir ADD / UPDATE / NOOP / INVALIDATE -> guardar created_at, valid_from, valid_to.
    Solo se guardan los campos `text` de los hechos: nunca el `analysis` del modelo.
  - read_memories: relevancia (embedder) + recencia (timestamps SIMULADOS) + importancia,
    con min-max e informe del rango de cada término antes de normalizar.
  - memory_filter: los hechos del usuario que coinciden con metadatos del corpus (doc_id,
    fecha de publicación) se convierten en filtros de la recuperación.
  - build_probe_prompt: el prompt de una sonda con un espacio de memoria y/o historial,
    dentro de presupuestos declarados y del tope num_ctx - num_predict - margen.
  - score_probe, audit_records, ability_table: métricas sin juez.

No modifica la evaluación congelada ni los artefactos de etapas anteriores.
"""
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Callable, Literal
import json
import re
import unicodedata

from pydantic import BaseModel, Field

from grounded_stage4 import GroundedAnswer, assemble, contains_key, passage_block, render_passages


# --- configuración ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Stage6Config:
    """Todo lo que decide qué significa una fila de resultados (va al run_spec)."""
    model: str = "qwen3:8b"
    num_ctx: int = 8192              # fijo, igual que en las Etapas 4 y 5
    num_predict: int = 700           # reserva de salida de la respuesta a la sonda
    safety_margin: int = 64
    passage_budget: int = 1500       # pasajes del corpus, en tokens de Qwen3
    memory_budget: int = 600         # espacio de memoria (§7.3: 300-1.000)
    history_budget: int = 3000       # reproducción: "todo lo que entre" dentro de este tope
    memory_k: int = 6                # recuerdos por sonda (§7.3: 3-10)
    recency_decay: float = 0.995     # por hora, sobre timestamps simulados (§7.3)
    probe_offset_hours: int = 24     # la sonda se hace en una sesión nueva, un día después
    write_candidates: int = 3        # recuerdos parecidos que ve el decisor
    write_similarity: float = 0.55   # por debajo (y con otro subject) se agrega sin consultar al LLM
    extract_num_predict: int = 800
    decide_num_predict: int = 300
    sweep_fractions: tuple = (0.5, 0.25)
    temperature: float = 0.0
    seed: int = 0

    def __post_init__(self):
        if self.num_ctx <= self.num_predict:
            raise ValueError("num_ctx tiene que ser mayor que num_predict.")
        if self.passage_budget + self.memory_budget >= self.prompt_limit:
            raise ValueError("Pasajes + memoria no dejan lugar para sistema y pregunta.")
        if self.temperature != 0.0:
            raise ValueError("La etapa fija temperatura 0.")
        if not 0 < self.recency_decay < 1:
            raise ValueError("recency_decay en (0, 1).")

    @property
    def prompt_limit(self) -> int:
        return self.num_ctx - self.num_predict - self.safety_margin


# --- esquemas ----------------------------------------------------------------------------------
class CandidateFact(BaseModel):
    subject: str = Field(max_length=60, description="clave corta del atributo, snake_case")
    kind: Literal["semantic", "episodic", "procedural"]
    text: str = Field(max_length=240, description="afirmación autocontenida, solo lo que se dijo")
    importance: int = Field(ge=1, le=10)


class Extraction(BaseModel):
    # "analysis" < "facts": el razonamiento sale primero, y NUNCA se guarda.
    analysis: str = Field(max_length=400)
    facts: list[CandidateFact] = Field(max_length=8)


class WriteDecision(BaseModel):
    # El razonamiento declarado primero, obligatorio, y con un nombre que también quede primero
    # ("analysis" < "decision" < "target_id" < "text").
    analysis: str = Field(max_length=400)
    decision: Literal["ADD", "UPDATE", "NOOP", "INVALIDATE"]
    target_id: str | None = None
    text: str | None = None


class MemoryRecord(BaseModel):
    memory_id: str
    kind: Literal["semantic", "episodic", "procedural"]
    subject: str                       # un único valor ACTUAL por subject
    text: str
    created_at: datetime               # cuándo lo escribió el sistema (reloj real)
    valid_from: datetime               # cuándo pasó a ser cierto (tiempo SIMULADO de la sesión)
    valid_to: datetime | None = None   # se setea al invalidar; el registro se queda
    source_session: str
    importance: int = Field(default=5, ge=1, le=10)
    supersedes: str | None = None      # el registro que reemplazó (UPDATE)


def check_reasoning_first(*schemas) -> None:
    for s in schemas:
        props = list(s.model_json_schema()["properties"])
        if props[0] != "analysis" or sorted(props)[0] != "analysis":
            raise AssertionError(f"{s.__name__}: el razonamiento tiene que ir primero: {props}")


# --- prompts -------------------------------------------------------------------------------------
SYSTEM_EXTRACT = """Sos el módulo de memoria de un asistente que conversa con un mismo usuario a lo
largo de varias sesiones. Leé la transcripción de UNA sesión y extraé los hechos sobre el usuario
que valga la pena recordar en sesiones futuras.

Reglas:
- Solo lo que el usuario dijo o confirmó explícitamente en ESTA sesión. Nada de hipótesis,
  inferencias ni suposiciones ("probablemente", "parece").
- No omitas datos: cada nombre propio (tiendas, academias, productos), importe, fecha,
  canal e identificador de norma que el usuario mencione tiene que quedar en algún hecho,
  junto con lo que el usuario pidió hacer con él.
- Un hecho por atributo: no combines datos distintos en una misma oración.
- Cada hecho es una oración autocontenida en tercera persona ("El usuario ..."), con los datos
  exactos (nombres, importes, fechas, identificadores de normas) tal como se dijeron.
- Si el usuario corrige o cambia un dato, el hecho es el valor NUEVO, y conviene decir que
  reemplaza al anterior.
- subject: una clave corta en snake_case que identifique el atributo (por ejemplo
  canal_contacto, precio_compra, costo_envio, tienda_compra, norma_referencia,
  formato_respuesta). Si el atributo ya existe en la lista de subjects guardados, usá el mismo.
- kind: "procedural" para preferencias sobre cómo responder; "episodic" para eventos (una compra,
  un pago); "semantic" para el resto de los hechos sobre el usuario.
- importance: de 1 a 10.

Devolvé un JSON con analysis (breve) y facts."""

SYSTEM_DECIDE = """Sos el módulo de memoria de un asistente. Hay un hecho NUEVO extraído de la
última sesión y una lista de recuerdos YA GUARDADOS parecidos. Decidí qué hacer con el almacén:

- ADD: el hecho nuevo es información que no está guardada.
- UPDATE: el hecho nuevo da un valor distinto para el mismo atributo que un recuerdo guardado
  (un cambio o una corrección). target_id = el memory_id de ese recuerdo; text = el hecho nuevo.
  El recuerdo viejo no se borra: queda como "ya no vigente".
- NOOP: SOLO si un recuerdo guardado dice exactamente lo mismo (mismo atributo y mismo valor).
  Si el hecho nuevo agrega cualquier dato que no está guardado (un nombre, un lugar, un
  producto, una preferencia), es ADD.
- INVALIDATE: el usuario dijo que un recuerdo guardado dejó de ser cierto, sin dar un valor
  nuevo. target_id = el memory_id de ese recuerdo.

Devolvé un JSON con analysis (breve), decision, target_id y text."""

SYSTEM_PROBE = """Sos un asistente que conversa con un mismo usuario a lo largo de varias sesiones y
responde preguntas sobre normas argentinas de defensa del consumidor.

Podés recibir:
- MEMORIA: recuerdos guardados de sesiones anteriores, cada uno con su vigencia.
- HISTORIAL: los mensajes de sesiones anteriores.
- PASAJES: fragmentos del corpus normativo, delimitados por [doc_id · section_id].

Los datos del usuario salen solo de la MEMORIA o del HISTORIAL; la normativa sale solo de los
PASAJES. Si un dato cambió, el valor actual es el más reciente; si te preguntan por el valor
anterior, usá el que ya no está vigente. No inventes datos del usuario que no figuren.

Devolvé un objeto JSON con estos campos, en este orden:
- analysis: dos o tres oraciones sobre de dónde sale la respuesta.
- answer: la respuesta, breve y directa, en español. Copiá nombres, importes y fechas tal como
  figuran. Si la pregunta es de sí o no, empezá con "Sí" o "No". Si no tenés la información,
  escribí exactamente: "No tengo información suficiente para responder."
- can_answer: true solo si la memoria, el historial o los pasajes contienen la respuesta.
- citations: los section_id de los pasajes que usaste (lista vacía si respondiste solo con datos
  del usuario).
- verdict: "si" o "no" si la pregunta se responde con sí o no; "no_aplica" en otro caso."""


# --- utilidades de texto ---------------------------------------------------------------------------
def _norm(text: str) -> str:
    text = unicodedata.normalize("NFD", (text or "").lower())
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"(?<=\d)[.,\s](?=\d{3}(?:\D|$))", "", text)
    return " ".join(re.sub(r"[^\w\s]", " ", text).split())


def verdict_from_answer(answer: str) -> str | None:
    """'si'/'no' según la primera palabra de la respuesta, que el prompt obliga a ser Sí/No.

    Igual que la corrección final de la Etapa 4: el campo `verdict` del schema resultó
    inconsistente con el propio texto del modelo, así que se lee el texto pedido.
    """
    words = _norm(answer).split()
    return words[0] if words and words[0] in ("si", "no") else None


def parse_dt(value) -> datetime:
    return value if isinstance(value, datetime) else datetime.fromisoformat(str(value))


def probe_time(session_date, cfg: Stage6Config) -> datetime:
    """La sonda se hace en una sesión NUEVA posterior a after_session (tiempo simulado)."""
    return parse_dt(session_date) + timedelta(hours=cfg.probe_offset_hours)


def session_transcript(session: dict) -> str:
    lines = []
    for t in session["turns"]:
        lines.append(f"Usuario: {t['user']}")
        if t.get("assistant"):
            lines.append(f"Asistente: {t['assistant']}")
    return "\n".join(lines)


# --- el almacén --------------------------------------------------------------------------------------
class MemoryStore:
    """Registros + vectores + log de operaciones. Nada se borra: se invalida.

    `embed(list[str]) -> np.ndarray` devuelve vectores normalizados (coseno = producto punto).
    `now()` da el reloj real para created_at (inyectable en las pruebas).
    """

    def __init__(self, scenario_id: str, embed: Callable, now: Callable[[], datetime] | None = None):
        self.scenario_id = scenario_id
        self.embed = embed
        self.now = now or (lambda: datetime.now().replace(microsecond=0))
        self.records: list[MemoryRecord] = []
        self.vectors: dict[str, object] = {}
        self.oplog: list[dict] = []

    # consultas
    def get(self, memory_id: str) -> MemoryRecord | None:
        return next((r for r in self.records if r.memory_id == memory_id), None)

    def valid(self) -> list[MemoryRecord]:
        return [r for r in self.records if r.valid_to is None]

    def subjects(self) -> list[str]:
        return sorted({r.subject for r in self.valid()})

    def similar(self, text: str, subject: str, k: int, threshold: float) -> list[tuple[MemoryRecord, float]]:
        """Recuerdos VIGENTES parecidos: mismo subject, o similitud >= threshold."""
        current = self.valid()
        if not current:
            return []
        q = self.embed([text])[0]
        scored = [(r, float(self.vectors[r.memory_id] @ q)) for r in current]
        keep = [(r, s) for r, s in scored if s >= threshold or r.subject == subject]
        return sorted(keep, key=lambda x: (-(x[0].subject == subject), -x[1]))[:k]

    # escrituras
    def _new_id(self, session_id: str) -> str:
        return f"{self.scenario_id}-{session_id}-{sum(r.source_session == session_id for r in self.records) + 1:02d}"

    def add(self, *, subject, kind, text, importance, session_id, session_date, supersedes=None) -> MemoryRecord:
        rec = MemoryRecord(memory_id=self._new_id(session_id), kind=kind, subject=subject, text=text,
                           created_at=self.now(), valid_from=parse_dt(session_date),
                           source_session=session_id, importance=importance, supersedes=supersedes)
        self.records.append(rec)
        self.vectors[rec.memory_id] = self.embed([text])[0]
        return rec

    def invalidate(self, memory_id: str, session_date) -> MemoryRecord:
        rec = self.get(memory_id)
        if rec is None or rec.valid_to is not None:
            raise KeyError(memory_id)
        rec.valid_to = parse_dt(session_date)
        return rec

    def log(self, **op) -> dict:
        op = {"op_n": len(self.oplog) + 1, "scenario_id": self.scenario_id, **op}
        self.oplog.append(op)
        return op

    def dump(self) -> list[dict]:
        return [r.model_dump(mode="json") for r in self.records]


def write_session(store: MemoryStore, session: dict, call: Callable, cfg: Stage6Config) -> dict:
    """El camino de escritura para una sesión. `call(messages, schema, num_predict)` devuelve
    (objeto o None, ChatResult o None, error). Devuelve el costo de la sesión."""
    sid, sdate = session["session_id"], session["date"]
    stats = {"session_id": sid, "llm_calls": 0, "prompt_tokens": 0, "output_tokens": 0,
             "ops": {"ADD": 0, "UPDATE": 0, "NOOP": 0, "INVALIDATE": 0}, "errors": 0,
             "think_leaks": 0, "facts": 0, "added_without_llm": 0}

    def account(r):
        stats["llm_calls"] += 1
        if r is not None:
            stats["prompt_tokens"] += r.prompt_tokens or 0
            stats["output_tokens"] += r.output_tokens or 0
            stats["think_leaks"] += int(bool(r.think_leak))

    messages = [{"role": "system", "content": SYSTEM_EXTRACT},
                {"role": "user", "content": (
                    f"Fecha de la sesión: {sdate}\n"
                    f"Subjects ya guardados: {', '.join(store.subjects()) or '(ninguno)'}\n\n"
                    f"TRANSCRIPCIÓN DE LA SESIÓN {sid}:\n{session_transcript(session)}")}]
    extraction, r, error = call(messages, Extraction, cfg.extract_num_predict)
    account(r)
    if error or extraction is None:
        stats["errors"] += 1
        store.log(session_id=sid, step="extract", error=error or "sin extracción")
        return stats
    store.log(session_id=sid, step="extract", n_facts=len(extraction.facts),
              facts=[f.model_dump() for f in extraction.facts], analysis_discarded=extraction.analysis)
    stats["facts"] = len(extraction.facts)

    for fact in extraction.facts:
        neighbours = store.similar(fact.text, fact.subject, cfg.write_candidates, cfg.write_similarity)
        # Los hechos de una misma extracción ya vienen separados por atributo: solo se comparan con
        # recuerdos de sesiones anteriores, o del mismo subject (decidido en dev: el decisor marcaba
        # NOOP la academia contra el precio recién guardado de la misma sesión, y se perdía).
        neighbours = [(n, s) for n, s in neighbours if n.source_session != sid or n.subject == fact.subject]
        base = {"session_id": sid, "step": "write", "candidate": fact.model_dump(),
                "neighbours": [{"memory_id": n.memory_id, "subject": n.subject, "text": n.text,
                                "similarity": round(s, 3)} for n, s in neighbours]}
        if not neighbours:
            rec = store.add(subject=fact.subject, kind=fact.kind, text=fact.text, importance=fact.importance,
                            session_id=sid, session_date=sdate)
            stats["ops"]["ADD"] += 1
            stats["added_without_llm"] += 1
            store.log(**base, decision="ADD", via="sin recuerdos parecidos (sin LLM)", new_id=rec.memory_id)
            continue
        listing = "\n".join(f"- memory_id={n.memory_id} | subject={n.subject} | vigente desde "
                            f"{n.valid_from.date()} | {n.text}" for n, _ in neighbours)
        dmsgs = [{"role": "system", "content": SYSTEM_DECIDE},
                 {"role": "user", "content": (
                     f"Fecha de la sesión: {sdate}\n\nHECHO NUEVO (subject={fact.subject}): {fact.text}\n\n"
                     f"RECUERDOS GUARDADOS PARECIDOS:\n{listing}")}]
        decision, r, error = call(dmsgs, WriteDecision, cfg.decide_num_predict)
        account(r)
        ids = {n.memory_id for n, _ in neighbours}
        verdict, flag = (decision.decision if decision else "ADD"), None
        if error or decision is None:
            verdict, flag = "ADD", f"decisión fallida ({error}): se agrega"
            stats["errors"] += 1
        elif verdict in ("UPDATE", "INVALIDATE") and decision.target_id not in ids:
            verdict, flag = "ADD", f"target_id inválido ({decision.target_id}): se agrega"
            stats["errors"] += 1
        op = {**base, "decision": verdict, "llm_decision": decision.decision if decision else None,
              "target_id": decision.target_id if decision else None,
              "analysis_discarded": decision.analysis if decision else None, "flag": flag}
        if verdict == "ADD":
            rec = store.add(subject=fact.subject, kind=fact.kind, text=fact.text, importance=fact.importance,
                            session_id=sid, session_date=sdate)
            op["new_id"] = rec.memory_id
        elif verdict == "UPDATE":
            old = store.invalidate(decision.target_id, sdate)
            # Se guarda el texto del hecho (o el del decisor), nunca un análisis.
            rec = store.add(subject=old.subject, kind=fact.kind, text=(decision.text or fact.text),
                            importance=fact.importance, session_id=sid, session_date=sdate,
                            supersedes=old.memory_id)
            op["new_id"], op["invalidated_id"] = rec.memory_id, old.memory_id
        elif verdict == "INVALIDATE":
            store.invalidate(decision.target_id, sdate)
            op["invalidated_id"] = decision.target_id
        stats["ops"][verdict] += 1
        store.log(**op)
    return stats


# --- el camino de lectura ------------------------------------------------------------------------------
def score(recency: list[float], importance: list[float], relevance: list[float]) -> list[float]:
    """La combinación del enunciado: min-max por término y suma."""
    def minmax(xs):
        if not xs:
            return []
        lo, hi = min(xs), max(xs)
        if hi - lo < 1e-6:          # solo si es EXACTAMENTE constante
            return [0.0] * len(xs)
        return [(x - lo) / (hi - lo) for x in xs]
    return [a + b + c for a, b, c in zip(minmax(recency), minmax(importance), minmax(relevance))]


def memory_line(rec: MemoryRecord) -> str:
    if rec.valid_to is None:
        status = f"vigente desde {rec.valid_from.date()}, sesión {rec.source_session}"
    else:
        status = (f"YA NO VIGENTE: valió del {rec.valid_from.date()} al {rec.valid_to.date()}, "
                  f"sesión {rec.source_session}")
    return f"- [{status}] ({rec.kind}, {rec.subject}) {rec.text}"


def read_memories(store: MemoryStore, question: str, when: datetime, cfg: Stage6Config,
                  count: Callable[[str], int]) -> dict:
    """Top-k por relevancia + recencia + importancia, dentro de memory_budget.

    Incluye registros invalidados (marcados como tales): una pregunta "¿qué había dicho antes?"
    los necesita. La recencia usa valid_from (tiempo simulado), nunca el reloj real.
    """
    recs = [r for r in store.records if r.valid_from <= when]
    if not recs:
        return {"selected": [], "lines": [], "tokens": 0, "ranges": {}, "scores": []}
    q = store.embed([question])[0]
    relevance = [float(store.vectors[r.memory_id] @ q) for r in recs]
    hours = [max(0.0, (when - r.valid_from).total_seconds() / 3600) for r in recs]
    recency = [cfg.recency_decay ** h for h in hours]
    importance = [float(r.importance) for r in recs]
    ranges = {name: (round(min(v), 4), round(max(v), 4))
              for name, v in (("relevance", relevance), ("recency", recency), ("importance", importance))}
    total = score(recency, importance, relevance)
    order = sorted(range(len(recs)), key=lambda i: (-total[i], recs[i].memory_id))[:cfg.memory_k]
    as_passages = [{"chunk_id": recs[i].memory_id, "doc_id": "MEM", "section_id": recs[i].memory_id,
                    "index_text": memory_line(recs[i]), "rec": recs[i]} for i in order]
    kept, _, _ = assemble(as_passages, cfg.memory_budget, count, render=lambda ps: "\n".join(p["index_text"] for p in ps))
    chosen = sorted((p["rec"] for p in kept), key=lambda r: (r.valid_from, r.memory_id))   # cronológico
    lines = [memory_line(r) for r in chosen]
    return {"selected": chosen, "lines": lines, "tokens": count("\n".join(lines)) if lines else 0,
            "ranges": ranges,
            "scores": [{"memory_id": recs[i].memory_id, "total": round(total[i], 4), "relevance": round(relevance[i], 4),
                        "recency": round(recency[i], 4), "importance": importance[i]} for i in order]}


# --- la memoria como filtro de la recuperación ----------------------------------------------------------
DOC_ID_RE = re.compile(r"\b[A-Z]{2,5}-[0-9][\w\-/]*")
DATE_RE = re.compile(r"\b(\d{4}-\d{2}-\d{2})\b")


def memory_filter(records: list[MemoryRecord], known_doc_ids: set[str], docs_by_date: dict[str, list[str]]) -> dict:
    """De los recuerdos VIGENTES: los doc_id del corpus que el usuario nombró, y las fechas que
    coinciden con metadata.date (la fecha de publicación registrada, no la vigencia)."""
    doc_ids, dates, sources = set(), set(), []
    for r in records:
        if r.valid_to is not None:
            continue
        found = {d for d in DOC_ID_RE.findall(r.text) if d in known_doc_ids}
        found_dates = {d for d in DATE_RE.findall(r.text) if d in docs_by_date}
        if found or found_dates:
            sources.append(r.memory_id)
        doc_ids |= found
        dates |= found_dates
    date_docs = sorted({d for x in dates for d in docs_by_date[x]})
    return {"doc_ids": sorted(doc_ids) or None, "dates": sorted(dates) or None,
            "date_doc_ids": date_docs or None, "source_records": sources}


def retrieve_filtered(retriever, query: str, allowed_doc_ids: set[str] | None) -> list[dict]:
    """El ranking BM25 de la Etapa 3 restringido a ciertos documentos.

    Sin filtro devuelve lo mismo que retriever.retrieve. Con filtro puntúa TODOS los chunks y se
    queda con los de esos documentos (si se filtrara el top-100 ya cortado, un documento con la
    respuesta en el puesto 101 desaparecería).
    """
    if not allowed_doc_ids:
        return retriever.retrieve(query)
    words = retriever.words(query)
    if not words:
        return []
    idx, sc = retriever.index.retrieve(retriever.tokenize([" ".join(words)]), k=len(retriever.chunks),
                                       show_progress=False)
    ranked = [{**retriever.chunks[int(i)], "score": float(s)} for i, s in zip(idx[0], sc[0])
              if float(s) > 0 and retriever.chunks[int(i)]["doc_id"] in allowed_doc_ids]
    return sorted(ranked, key=lambda c: (-c["score"], c["chunk_id"]))[:retriever.cfg.candidate_k]


# --- historial (reproducción) ----------------------------------------------------------------------------
def session_messages(session: dict) -> list[dict]:
    msgs = []
    for j, t in enumerate(session["turns"]):
        prefix = f"[Sesión {session['session_id']} · {session['date']}]\n" if j == 0 else ""
        msgs.append({"role": "user", "content": prefix + t["user"]})
        if t.get("assistant"):
            msgs.append({"role": "assistant", "content": t["assistant"]})
    return msgs


def replay_history(sessions: list[dict], budget: int | None, count: Callable[[str], int]) -> dict:
    """Las sesiones pasadas, enteras, de la más reciente hacia atrás, mientras entren en `budget`.

    budget=None: todas (el brazo de reproducción completa verifica después que entren en num_ctx).
    """
    kept, used, full = [], 0, 0
    sizes = {s["session_id"]: sum(count(f"{m['role']}: {m['content']}") for m in session_messages(s)) for s in sessions}
    full = sum(sizes.values())
    for s in reversed(sessions):
        if budget is not None and used + sizes[s["session_id"]] > budget:
            break
        kept.insert(0, s)
        used += sizes[s["session_id"]]
    return {"messages": [m for s in kept for m in session_messages(s)], "sessions": [s["session_id"] for s in kept],
            "tokens": used, "full_tokens": full, "session_tokens": sizes,
            "dropped": [s["session_id"] for s in sessions if s not in kept]}


# --- el prompt de una sonda ------------------------------------------------------------------------------
def probe_user_message(question: str, when: datetime, memory_lines: list[str] | None, passages: list[dict]) -> str:
    parts = [f"Fecha de hoy: {when.date()} (sesión nueva)."]
    if memory_lines is not None:
        parts.append("MEMORIA:\n" + ("\n".join(memory_lines) if memory_lines else "(no hay recuerdos guardados)"))
    parts.append("PASAJES:\n" + (render_passages(passages) if passages else "(no se entregó ningún pasaje)"))
    parts.append(f"PREGUNTA: {question}")
    return "\n\n".join(parts)


def build_probe_prompt(question: str, when: datetime, ranked: list[dict], cfg: Stage6Config,
                       count: Callable[[str], int], count_messages: Callable[[list[dict]], int | None],
                       memory_lines: list[str] | None = None, history: list[dict] | None = None) -> dict:
    """Sistema + historial + (memoria + pasajes + pregunta). Los pasajes se llenan hasta
    passage_budget con el ensamblador de la Etapa 4 (saltea y sigue); si el prompt completo no
    entra en el tope, se saca el último pasaje (nunca la memoria ni el historial: si eso no
    entra, la fila es 'no factible' y lo decide quien llama)."""
    delivered, skipped, ptoks = assemble(ranked, cfg.passage_budget, count) if ranked else ([], [], 0)
    while True:
        msgs = ([{"role": "system", "content": SYSTEM_PROBE}] + list(history or [])
                + [{"role": "user", "content": probe_user_message(question, when, memory_lines, delivered)}])
        expected = count_messages(msgs)
        if expected is None or expected <= cfg.prompt_limit or not delivered:
            break
        skipped.append(delivered.pop())
        ptoks = count(render_passages(delivered)) if delivered else 0
    return {"messages": msgs, "delivered": delivered, "passage_tokens": ptoks,
            "expected_prompt_tokens": expected, "fits": expected is None or expected <= cfg.prompt_limit}


# --- puntaje sin juez ------------------------------------------------------------------------------------
def score_probe(probe: dict, answer: GroundedAnswer | None, error: str | None, delivered: list[dict]) -> dict:
    gold = {(g["doc_id"], g.get("section_id")) for g in probe.get("gold") or []}
    m = {"answerable": probe["answerable"], "ability": probe["ability"],
         "gold_delivered": (any((p["doc_id"], p["section_id"]) in gold for p in delivered) if gold else None)}
    if error is not None or answer is None:
        m.update(outcome="error", error=error or "sin respuesta", correct=None)
        return m
    abstained = not answer.can_answer
    expected = probe.get("expected_verdict")
    verdict = verdict_from_answer(answer.answer)
    key_hit = contains_key(answer.answer, probe.get("answer_key")) if probe.get("answer_key") else None
    if not probe["answerable"]:
        correct = abstained
    elif expected:
        correct = (not abstained) and verdict == {"yes": "si", "no": "no"}[expected]
    else:
        correct = (not abstained) and bool(key_hit)
    m.update(outcome="abstained" if abstained else "answered", error=None, correct=bool(correct),
             key_hit=key_hit, verdict=verdict, expected_verdict=expected,
             cited=[c for c in answer.citations if c])
    return m


def evidence_recall(probe: dict, sessions_seen: list[str]) -> float | None:
    ev = probe.get("evidence_sessions") or []
    if not ev:
        return None
    return sum(e in set(sessions_seen) for e in ev) / len(ev)


# --- auditoría de lo guardado --------------------------------------------------------------------------
HEDGE_RE = re.compile(r"\b(probablemente|posiblemente|quizas|tal vez|parece|supongo|podria|seguramente|asumo|"
                      r"infiero|aparentemente|presumiblemente)\b")
NUM_RE = re.compile(r"\d+")


def audit_records(records: list[MemoryRecord], sessions: dict[str, dict], analyses: list[str],
                  fact_texts: list[str] = ()) -> list[dict]:
    """Chequeos automáticos de cada registro (la lectura humana va aparte):
    - think_tag: contiene <think> o </think>;
    - hedge: formulado como hipótesis ("probablemente", "parece"...);
    - analysis_leak: el texto es (parte de) un `analysis` que el modelo escribió;
    - unsaid_numbers: números que no aparecen en la transcripción de su sesión."""
    out = []
    norm_analyses = [_norm(a) for a in analyses if a]
    norm_facts = {_norm(t) for t in fact_texts if t}
    for r in records:
        text_n = _norm(r.text)
        source = _norm(session_transcript(sessions[r.source_session])) if r.source_session in sessions else ""
        nums = [n for n in NUM_RE.findall(text_n) if n not in NUM_RE.findall(source)]
        out.append({"memory_id": r.memory_id, "subject": r.subject, "text": r.text,
                    "valid": r.valid_to is None,
                    "think_tag": "<think>" in r.text or "</think>" in r.text,
                    "hedge": bool(HEDGE_RE.search(text_n)),
                    "analysis_leak": text_n not in norm_facts and any(text_n and (text_n in a) for a in norm_analyses),
                    "unsaid_numbers": nums})
    return out


def ability_table(rows: list[dict], arms: list[str], abilities: list[str]) -> list[dict]:
    """Conteos por habilidad y brazo ('3/4', no 75%). Los errores se cuentan aparte."""
    table = []
    for ab in abilities + ["total"]:
        line = {"ability": ab}
        for arm in arms:
            rs = [r for r in rows if r["arm"] == arm and (ab == "total" or r["metrics"]["ability"] == ab)]
            ok = sum(bool(r["metrics"]["correct"]) for r in rs if r["metrics"]["outcome"] != "error")
            err = sum(r["metrics"]["outcome"] == "error" for r in rs)
            line[arm] = f"{ok}/{len(rs)}" + (f" ({err} err)" if err else "")
        table.append(line)
    return table


# --- embedder de respaldo ---------------------------------------------------------------------------
def lexical_embedder(dim: int = 4096):
    """Bolsa de palabras con hashing, normalizada. NO es el embedder del proyecto (bge-m3): sirve
    para las pruebas sin GPU y como respaldo declarado si bge-m3 no se puede cargar."""
    import hashlib as _h
    import numpy as np

    def embed(texts):
        out = np.zeros((len(texts), dim), dtype=np.float32)
        for i, t in enumerate(texts):
            for w in _norm(t).split():
                if len(w) > 2:
                    out[i, int(_h.md5(w[:6].encode()).hexdigest(), 16) % dim] += 1.0
            n = float(np.linalg.norm(out[i]))
            if n:
                out[i] /= n
        return out
    return embed


def to_jsonable(x):
    return json.loads(json.dumps(x, ensure_ascii=False, default=str))


IMPLEMENTATION_SHA256 = 'cfd44c28cc055e6da6f78a3c2fee4b94142428904f6f178f063fd7d7523fa4cd'

In [7]:
CFG = Stage6Config(model=MODEL)
check_reasoning_first(Extraction, WriteDecision, GroundedAnswer)
print(json.dumps(asdict(CFG), indent=2), '\nprompt_limit:', CFG.prompt_limit)

{
  "model": "qwen3:8b",
  "num_ctx": 8192,
  "num_predict": 700,
  "safety_margin": 64,
  "passage_budget": 1500,
  "memory_budget": 600,
  "history_budget": 3000,
  "memory_k": 6,
  "recency_decay": 0.995,
  "probe_offset_hours": 24,
  "write_candidates": 3,
  "write_similarity": 0.55,
  "extract_num_predict": 800,
  "decide_num_predict": 300,
  "sweep_fractions": [
    0.5,
    0.25
  ],
  "temperature": 0.0,
  "seed": 0
} 
prompt_limit: 7428


In [9]:
from pathlib import Path
import shutil
import json

base = Path("/content/drive/MyDrive/projecto_rag")

origen = base / "artifacts_respaldo_20261101_215934/full/index/stage3_selected_retriever.json"
destino = base / "artifacts/full/index/stage3_selected_retriever.json"

if not origen.is_file():
    raise FileNotFoundError(f"No encontré el archivo de respaldo: {origen}")

destino.parent.mkdir(parents=True, exist_ok=True)

if not destino.exists():
    shutil.copy2(origen, destino)
    print("Archivo restaurado correctamente.")
else:
    print("El archivo ya existe; no se sobrescribió.")

seleccion = json.loads(destino.read_text(encoding="utf-8"))

print("Recuperador seleccionado:", seleccion["selected_arm"])
print("Índice esperado:", Path(seleccion["index_dir"]).name)

indice = destino.parent / Path(seleccion["index_dir"]).name
print("¿Está chunks.jsonl?", (indice / "chunks.jsonl").is_file())
print("¿Está manifest.json?", (indice / "manifest.json").is_file())

El archivo ya existe; no se sobrescribió.
Recuperador seleccionado: bm25
Índice esperado: stage3_36a95c9c1551289f
¿Está chunks.jsonl? True
¿Está manifest.json? True


In [10]:
from pathlib import Path
import json
import shutil
from grounded_stage4 import canonical_hash, read_jsonl

base = Path("/content/drive/MyDrive/projecto_rag")

# Manifiesto del índice seleccionado originalmente
manifest_src = (
    base / "artifacts_respaldo_20261101_215934"
    / "full/stage3_36a95c9c1551289f/manifest.json"
)

# Chunks de una ejecución anterior de la Etapa 3
chunks_src = (
    base / "artifacts/full/index"
    / "stage3_3527c9a3b1c33e08/chunks.jsonl"
)

destino = (
    base / "artifacts/full/index"
    / "stage3_36a95c9c1551289f"
)

assert manifest_src.is_file(), f"Falta: {manifest_src}"
assert chunks_src.is_file(), f"Falta: {chunks_src}"

manifest = json.loads(manifest_src.read_text(encoding="utf-8"))
chunks = read_jsonl(chunks_src)

hash_obtenido = canonical_hash(chunks)
hash_esperado = manifest["chunks_hash"]

print("Chunks encontrados:", len(chunks))
print("Chunks esperados:", manifest["chunks"])
print("Hash coincide:", hash_obtenido == hash_esperado)

if len(chunks) == manifest["chunks"] and hash_obtenido == hash_esperado:
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copy2(manifest_src, destino / "manifest.json")
    shutil.copy2(chunks_src, destino / "chunks.jsonl")
    print("Índice BM25 restaurado y validado correctamente.")
else:
    print("Los archivos no coinciden. No se copió nada.")

Chunks encontrados: 13630
Chunks esperados: 13630
Hash coincide: True
Índice BM25 restaurado y validado correctamente.


In [11]:
import schema
import llm
from console import StageRun, print_table, verdict, clip, wrap
from store import ResultsLog, write_json, write_jsonl
from grounded_stage4 import canonical_hash, read_jsonl


def local_or_drive(rel: str) -> Path:
    """artifacts/full/<rel>; si falta y Drive está montado, se copia desde Drive (como la Etapa 5)."""
    local, remote = ART_FULL / rel, DRIVE_ART / 'full' / rel
    if not local.exists() and remote.exists():
        if remote.is_dir():
            shutil.copytree(remote, local, dirs_exist_ok=True)
        else:
            local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(remote, local)
        print('copiado desde Drive:', rel)
    if not local.exists():
        raise FileNotFoundError(f'Falta {local} (ni en artifacts/ ni en Drive).')
    return local


local_or_drive('documents.jsonl')
schema.validate_documents(DOC_PATH).show().raise_if_failed()
EV = schema.load_eval(EVAL_DIR, documents_path=DOC_PATH)          # hashes de los 3 archivos + corpus
schema.validate_eval(EVAL_DIR, DOC_PATH).show().raise_if_failed()
FROZEN = json.loads((EVAL_DIR / 'FROZEN.json').read_text(encoding='utf-8'))

SCENARIOS = [s for s in read_jsonl(EVAL_DIR / 'scenarios.jsonl') if s['split'] == SPLIT]
assert SCENARIOS, f'No hay escenarios para {SPLIT}'
if SMOKE_TEST:   # §7.4: un escenario de dos sesiones con tres sondas
    s0 = dict(SCENARIOS[0])
    s0['sessions'] = s0['sessions'][:2]
    keep = {s['session_id'] for s in s0['sessions']}
    s0['probes'] = [p for p in s0['probes'] if p['after_session'] in keep][:3]
    SCENARIOS = [s0]

DOCS = {d['doc_id']: d for d in read_jsonl(DOC_PATH)}
KNOWN_DOC_IDS = set(DOCS)
DOCS_BY_DATE = {}
for d in DOCS.values():
    if d['metadata'].get('date'):
        DOCS_BY_DATE.setdefault(d['metadata']['date'], []).append(d['doc_id'])

# --- índice de la Etapa 3 ----------------------------------------------------------------------
# La elección de la Etapa 3: la copia de Drive tiene prioridad sobre la de git (puede estar más nueva).
_sel_drive = DRIVE_ART / 'full' / 'index' / 'stage3_selected_retriever.json'
selection_path = _sel_drive if _sel_drive.is_file() else local_or_drive('index/stage3_selected_retriever.json')
SELECTION = json.loads(selection_path.read_text(encoding='utf-8'))
if SELECTION['selected_arm'] != 'bm25':
    raise NotImplementedError(f"La Etapa 3 eligió {SELECTION['selected_arm']}; esta etapa reconstruye BM25. "
                              'Cargá ese recuperador con stage3_runtime.load_stage3.')
if SELECTION['corpus_hash'] != FROZEN['documents_sha256']:
    raise ValueError('El índice elegido en la Etapa 3 es de otro corpus.')


def find_index_dir():
    """El índice BM25 de la Etapa 3: STAGE3_INDEX_DIR, o el nombrado en la elección (en artifacts/ o en
    Drive), o cualquier índice stage3_* del mismo corpus con chunks.jsonl (en artifacts/, Drive o en
    cualquier carpeta de MyDrive hasta 4 niveles). Si hay varios, avisa y toma el más reciente."""
    if STAGE3_INDEX_DIR is not None:
        return Path(STAGE3_INDEX_DIR)
    name = Path(SELECTION['index_dir']).name
    for base in (ART_FULL / 'index', DRIVE_ART / 'full' / 'index'):
        if (base / name / 'chunks.jsonl').is_file():
            return base / name
    roots = [ART_FULL / 'index', DRIVE_ART / 'full' / 'index']
    found = []
    for root in roots:
        found += [p.parent for p in root.glob('stage3_*/chunks.jsonl')] if root.exists() else []
    mydrive = Path('/content/drive/MyDrive')
    if not found and mydrive.exists():
        for depth in ('*/', '*/*/', '*/*/*/', '*/*/*/*/'):
            found += [p.parent for p in mydrive.glob(depth + 'stage3_*/chunks.jsonl')]
    same_corpus = []
    for d in dict.fromkeys(found):
        try:
            if json.loads((d / 'manifest.json').read_text(encoding='utf-8')).get('corpus_hash') == SELECTION['corpus_hash']:
                same_corpus.append(d)
        except (OSError, ValueError):
            pass
    if not same_corpus:
        raise FileNotFoundError(
            f"No encontré el índice BM25 de la Etapa 3 ({name}) ni otro del mismo corpus, ni en artifacts/ ni en "
            "Drive. Si la Etapa 3 la corrió otra persona, el índice está en SU Drive: que comparta la carpeta "
            "projecto_rag/artifacts/full/index/" + name + " (y la agregues como acceso directo en tu MyDrive), o "
            "fijá STAGE3_INDEX_DIR con la ruta. Alternativa: correr notebooks/03_retrieval.ipynb (SMOKE_TEST=False) "
            "hasta la sección 9, que reconstruye y guarda el índice BM25.")
    same_corpus.sort(key=lambda d: (d / 'chunks.jsonl').stat().st_mtime)
    if len(same_corpus) > 1 or same_corpus[-1].name != name:
        print(f'AVISO: no está {name}; uso {same_corpus[-1]} (mismo corpus). Encontrados: {[str(d) for d in same_corpus]}')
    return same_corpus[-1]


INDEX_DIR = find_index_dir()
print('Índice de la Etapa 3:', INDEX_DIR)




INDEX_MANIFEST = json.loads((INDEX_DIR / 'manifest.json').read_text(encoding='utf-8'))
CHUNKS = read_jsonl(INDEX_DIR / 'chunks.jsonl')
if canonical_hash(CHUNKS) != INDEX_MANIFEST['chunks_hash']:
    raise ValueError('chunks.jsonl no coincide con el manifiesto del índice de la Etapa 3.')

# La misma clase BM25Retriever del notebook 03 (igual que la Etapa 5).
nb3 = json.loads(Path('notebooks/03_retrieval.ipynb').read_text(encoding='utf-8'))
cells3 = []
for c in nb3['cells']:
    if c['cell_type'] != 'code':
        continue
    src = ''.join(c['source'])
    try:
        names = {n.name for n in ast.parse(src).body if isinstance(n, ast.ClassDef)}
    except SyntaxError:
        continue
    if {'BM25Retriever', 'RetrievalConfig'} <= names:
        cells3.append(src)
if len(cells3) != 1:
    raise RuntimeError(f'Esperaba una celda con BM25Retriever en el notebook 03; hay {len(cells3)}.')
mod3 = types.ModuleType('stage3_bm25_embedded')
sys.modules['stage3_bm25_embedded'] = mod3
exec(compile(cells3[0], 'stage3_bm25_embedded', 'exec'), mod3.__dict__)
RETRIEVER = mod3.BM25Retriever(CHUNKS, mod3.RetrievalConfig(**INDEX_MANIFEST['config']))
STAGE3_CELL_SHA256 = hashlib.sha256(cells3[0].encode('utf-8')).hexdigest()

# Guarda: cada gold de las sondas tiene que estar en el índice (y un gold faltante, a propósito, se rechaza).
SECTIONS_IN_INDEX = {(c['doc_id'], c['section_id']) for c in CHUNKS}


def check_probe_gold(scenarios):
    missing = [(s['scenario_id'], p['probe_id'], g) for s in scenarios for p in s['probes']
               for g in p['gold'] if (g['doc_id'], g['section_id']) not in SECTIONS_IN_INDEX]
    if missing:
        raise ValueError(f'Gold de sondas ausente del índice: {missing}')


check_probe_gold(SCENARIOS)
try:
    check_probe_gold([{'scenario_id': 'guarda', 'probes': [{'probe_id': 'p0', 'gold': [
        {'doc_id': '__falta__', 'section_id': '__falta__#ART-1'}]}]}])
except ValueError:
    print('OK: un gold faltante se rechaza.')
else:
    raise AssertionError('No saltó la guarda de gold.')

pending = sum('pendiente' in p.get('notes', '').lower() for s in SCENARIOS for p in s['probes'])
print(f"Split {SPLIT} | escenarios: {[s['scenario_id'] for s in SCENARIOS]} | sondas: "
      f"{sum(len(s['probes']) for s in SCENARIOS)} | chunks: {len(CHUNKS)} | índice: {INDEX_DIR.name}")
if pending:
    print(f'Nota: {pending} sondas conservan "revisión humana pendiente" en notes (texto del borrador; '
          'la revisión está en eval/revision_manual.csv).')

documents (documents.jsonl): OK
   documents: 170
   sections: 7070
   characters: 10685267
   with version: 0
   with date: 170
eval set (eval): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12
Índice de la Etapa 3: /content/projecto_rag/artifacts/full/index/stage3_36a95c9c1551289f


DEBUG:bm25s:Building index from IDs objects


OK: un gold faltante se rechaza.
Split test | escenarios: ['m002', 'm003'] | sondas: 18 | chunks: 13630 | índice: stage3_36a95c9c1551289f
Nota: 13 sondas conservan "revisión humana pendiente" en notes (texto del borrador; la revisión está en eval/revision_manual.csv).


In [12]:
if EMBEDDER == 'bge-m3':
    import torch
    from sentence_transformers import SentenceTransformer
    EMB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    EMB_REVISION = SELECTION.get('dense_revision')
    _encoder = SentenceTransformer(SELECTION.get('dense_model', 'BAAI/bge-m3'), revision=EMB_REVISION,
                                   device=EMB_DEVICE)
    if EMB_DEVICE == 'cuda':
        _encoder = _encoder.half()
    _encoder.max_seq_length = 512

    def embed(texts):
        return _encoder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True,
                               batch_size=16, show_progress_bar=False)
    EMBEDDER_ID = {'model': SELECTION.get('dense_model', 'BAAI/bge-m3'), 'revision': EMB_REVISION,
                   'device': EMB_DEVICE, 'max_seq_length': 512}
else:
    embed = lexical_embedder()
    EMBEDDER_ID = {'model': 'lexical-hashing-bow', 'revision': None}
    warnings.warn('Embedder de respaldo: no es el de la Etapa 3. Declararlo en el informe.')

_v = embed(['El usuario prefiere el correo electrónico.', 'Canal de contacto preferido', 'Precio de la campera'])
print('Embedder:', EMBEDDER_ID, '| similitudes de prueba:', round(float(_v[0] @ _v[1]), 3), round(float(_v[0] @ _v[2]), 3))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedder: {'model': 'BAAI/bge-m3', 'revision': '5617a9f61b028005a4858fdac845db406aefb181', 'device': 'cuda', 'max_seq_length': 512} | similitudes de prueba: 0.697 0.314


In [14]:
environment.gpu_orphans()
INFO = llm.setup(MODEL, num_ctx=CFG.num_ctx)
print(llm.placement(MODEL))
llm.use_cache(ART / 'llm_cache' / f"{MODEL.replace(':', '_').replace('/', '_')}.jsonl")


def count(text):
    return llm.count_tokens(text, MODEL)


def count_messages(messages):
    return llm.count_prompt_tokens(messages, MODEL)


def call(messages, out_schema, num_predict):
    """(objeto o None, ChatResult o None, error) -- ContextOverflow atrapado y registrado."""
    try:
        obj, r = llm.structured(messages, out_schema, model=MODEL, num_ctx=CFG.num_ctx, num_predict=num_predict,
                                temperature=CFG.temperature, seed=CFG.seed, think=False)
    except llm.ContextOverflow as e:
        return None, None, f'CONTEXT_OVERFLOW (client): {e}'
    return obj, r, r.error


if count('hola') is None:
    raise RuntimeError(f'{MODEL} no tiene tokenizer en llm.HF_TOKENIZERS: las guardas estarían apagadas.')

   model        qwen3:8b
   digest       500a1f067a9f
   server       0.34.4
   num_ctx      8192
   thinking     {'honors_think_false': True, 'leaks_think_tag': False, 'thinking_only': False, 'supports_thinking': True, 'answer': 'Paris'}
   key_order    {'order': 'declaration', 'reply': '{"zeta": "alpha", "alfa": "omega"}', 'error': None}
   calibration  {'template_minus_server': [-4, -4, -4], 'offset': -4, 'repeat_prompt_tokens': (295, 295), 'repeat_cached_tokens': (3, 294), 'prompt_eval_count_includes_cached': True}
   placement    qwen3:8b: 100% GPU · 6.3 GB · context 8192
qwen3:8b: 100% GPU · 6.3 GB · context 8192
LLM call cache: /content/projecto_rag/artifacts/full/llm_cache/qwen3_8b.jsonl (0 entries)


## Mapeo técnico de las piezas

| Etapa 6 (manual) | LangGraph (7b) | Decisión conservada |
|---|---|---|
| `write_session()` | nodo `write` y `InMemoryStore` | mismos esquemas y ADD/UPDATE/NOOP/INVALIDATE |
| `MemoryStore.records` | items `InMemoryStore` por escenario | no borrar: invalidar con `valid_to` |
| Sesiones y sondas en bucle Python | invocaciones separadas por `thread_id` | cada sonda en sesión nueva; memoria compartida por escenario |
| `read_memories()` | nodo `prepare` | top-6, 600 tokens, recencia simulada |
| `RETRIEVER.retrieve` y `memory_filter` | nodo `prepare` | BM25 de Etapa 3 y filtro idéntico |
| `build_probe_prompt()` | nodo `prepare` | mismos 1.500 tokens para pasajes, `num_ctx=8192`, 700 salida, margen 64 |
| `call()`/`GroundedAnswer` | nodo `answer` | Qwen3:8b/Ollama, `think=False`, `temperature=0` |
| `score_probe` | nodo `answer` | mismo corrector de Etapa 6 |
| Estado local de una llamada | `InMemorySaver` por hilo | guardado automático de checkpoints, sin inyectar historial en prompt |

**Aviso de interpretabilidad:** `InMemoryStore` no selecciona automáticamente hechos, ni resume; la política de qué guardar y recordar sigue siendo la propia de Etapa 6. Su ventana no se usa por defecto: se fuerza explícitamente `num_ctx` en el cliente Ollama. El checkpointer conserva estado para reanudación en **la sesión de Colab**, no en disco después de reiniciar.

In [15]:
from typing import Any, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from datetime import datetime
import math

LG_STORE = InMemoryStore()
CHECKPOINTER = InMemorySaver()
ARMS_7B = ('memory', 'memory_filter')

class GraphState(TypedDict, total=False):
    phase: str
    scenario_id: str
    session: dict
    probe: dict
    arm: str
    write_stats: dict
    prepared: dict
    row: dict


def namespace(scenario_id: str) -> tuple[str, ...]:
    return ('stage7b', PROFILE, SPLIT, scenario_id, 'memory')


def load_from_langgraph(scenario_id: str):
    """Lee SOLO desde LangGraph Store, reconstruyendo vectores para usar las mismas
    funciones que en Etapa 6. Incluye recuerdos invalidados para preguntas históricas."""
    ledger = MemoryStore(scenario_id, embed)
    items = LG_STORE.search(namespace(scenario_id), limit=1000)
    ledger.records = [MemoryRecord.model_validate(item.value['record']) for item in items]
    ledger.records.sort(key=lambda rec: (rec.valid_from, rec.memory_id))
    if ledger.records:
        vecs = embed([rec.text for rec in ledger.records])
        ledger.vectors = {rec.memory_id: vec for rec, vec in zip(ledger.records, vecs)}
    return ledger


def save_into_langgraph(scenario_id: str, ledger):
    for rec in ledger.records:
        LG_STORE.put(namespace(scenario_id), rec.memory_id,
                     {'record': rec.model_dump(mode='json')})


def node_write(state: GraphState):
    sid = state['scenario_id']
    session = state['session']
    # Defensa contra ejecutar dos veces una escritura en el mismo proceso/Store
    meta_ns = ('stage7b', PROFILE, SPLIT, sid, 'completed_sessions')
    if LG_STORE.get(meta_ns, session['session_id']) is not None:
        raise RuntimeError(f'La sesión {sid}/{session["session_id"]} ya está escrita. Reiniciá el estado 7b antes de repetir.')
    ledger = load_from_langgraph(sid)
    stats = write_session(ledger, session, call, CFG)
    save_into_langgraph(sid, ledger)
    LG_STORE.put(meta_ns, session['session_id'], {'written': True, 'stats': stats})
    return {'write_stats': stats}


def node_prepare(state: GraphState):
    sid = state['scenario_id']
    probe = state['probe']
    arm = state['arm']
    assert arm in ARMS_7B
    ledger = load_from_langgraph(sid)
    sc = next(x for x in SCENARIOS if x['scenario_id'] == sid)
    dates = {s['session_id']: s['date'] for s in sc['sessions']}
    when = probe_time(dates[probe['after_session']], CFG)
    question = probe['question']
    mem = read_memories(ledger, question, when, CFG, count)
    filt = memory_filter(ledger.records, KNOWN_DOC_IDS, DOCS_BY_DATE)
    ranked = RETRIEVER.retrieve(question)
    if arm == 'memory_filter' and filt['doc_ids']:
        ranked = retrieve_filtered(RETRIEVER, question, set(filt['doc_ids']))
    p = build_probe_prompt(question, when, ranked, CFG, count, count_messages,
                           memory_lines=mem['lines'])
    return {'prepared': {
        'messages': p['messages'], 'fits': p['fits'],
        'expected_prompt_tokens': p['expected_prompt_tokens'],
        'passage_tokens': p['passage_tokens'],
        'delivered': [{'doc_id': x['doc_id'], 'section_id': x['section_id']} for x in p['delivered']],
        'memory_ids': [r.memory_id for r in mem['selected']],
        'memory_tokens': mem['tokens'], 'filter_doc_ids': filt['doc_ids'],
        'sessions_seen': [r.source_session for r in mem['selected']],
        'read_ranges': mem['ranges'],
    }}


def node_answer(state: GraphState):
    p, probe = state['prepared'], state['probe']
    item_id = f"{state['scenario_id']}/{probe['probe_id']}"
    if not p['fits']:
        answer, response, error = None, None, 'NOT_FEASIBLE: memoria/historial no entran en el tope'
    else:
        answer, response, error = call(p['messages'], GroundedAnswer, CFG.num_predict)
    metrics = score_probe(probe, answer, error, p['delivered'])
    row = {
        'stage': '07b_langgraph', 'arm': state['arm'], 'item_id': item_id,
        'scenario_id': state['scenario_id'], 'probe_id': probe['probe_id'],
        'ability': probe['ability'], 'after_session': probe['after_session'],
        'question': probe['question'], 'metrics': metrics, 'error': error,
        'memory_ids': p['memory_ids'], 'memory_tokens': p['memory_tokens'],
        'history_tokens': 0, 'history_sessions': None,
        'passage_tokens': p['passage_tokens'],
        'expected_prompt_tokens': p['expected_prompt_tokens'],
        'delivered': p['delivered'],
        'filter_doc_ids': p['filter_doc_ids'] if state['arm']=='memory_filter' else None,
        'sessions_seen': sorted(set(p['sessions_seen'])),
        'evidence_recall': evidence_recall(probe, p['sessions_seen']),
        'llm': response.log() if response is not None else None,
        'output': answer.model_dump() if answer is not None else None,
        'read_ranges': p['read_ranges'],
    }
    return {'row': row}


def route(state: GraphState):
    return state['phase']

builder = StateGraph(GraphState)
builder.add_node('write', node_write)
builder.add_node('prepare', node_prepare)
builder.add_node('answer', node_answer)
builder.add_conditional_edges(START, route, {'write': 'write', 'probe': 'prepare'})
builder.add_edge('write', END)
builder.add_edge('prepare', 'answer')
builder.add_edge('answer', END)
GRAPH = builder.compile(checkpointer=CHECKPOINTER, store=LG_STORE)
print('LangGraph compilado. Nodos: write / prepare / answer')

LangGraph compilado. Nodos: write / prepare / answer


## Ejecutar las mismas sondas

**Restricciones:** `scenarios.jsonl` congelado; ninguna pregunta ni *gold* se edita. Cada sesión se escribe una vez. Cada sonda abre un hilo nuevo con un `thread_id` único; todos los hilos del mismo escenario comparten el `InMemoryStore` de ese escenario.

Se ejecutan **2 brazos** (`memory`, `memory_filter`) y el camino de escritura por sesión. En `test` deberían ser **18 sondas × 2 = 36 respuestas** (más las llamadas de escritura). En `dev`, la corrida de prueba es solo diagnóstica.

In [16]:
from store import write_json, write_jsonl
from grounded_stage4 import canonical_hash

RUN_SPEC_7B = {
    'stage': '07b_langgraph', 'mode': 'functional_parity',
    'framework': 'langgraph', 'version': LANGGRAPH_VERSION,
    'split': SPLIT, 'profile': PROFILE, 'model': MODEL,
    'model_digest': INFO.get('digest'), 'ollama_server': INFO.get('server'),
    'config': asdict(CFG), 'num_ctx_explicit': CFG.num_ctx,
    'think_explicit': False, 'temperature': CFG.temperature,
    'embedder': EMBEDDER_ID, 'index': INDEX_DIR.name,
    'index_chunks_hash': INDEX_MANIFEST['chunks_hash'],
    'frozen_eval': EV.key(), 'prompts_sha256': canonical_hash({'extract': SYSTEM_EXTRACT, 'decide': SYSTEM_DECIDE, 'probe': SYSTEM_PROBE, 'schemas': [m.model_json_schema() for m in (Extraction, WriteDecision, GroundedAnswer)]}),
    'arms': list(ARMS_7B),
    'mem_semantics': 'Original stage6 write_session/read_memories; LangGraph persistence/orchestration only',
    'checkpointer': 'InMemorySaver, per-thread, not durable across Colab restarts',
    'store': 'InMemoryStore, cross-thread, not durable across Colab restarts',
    'summarizer': 'none',
}
KEY_7B = canonical_hash(RUN_SPEC_7B)[:16]
RESULTS_7B = ART / 'results' / ('stage7b_langgraph_' + KEY_7B)
RESULTS_7B.mkdir(parents=True, exist_ok=True)
write_json(RESULTS_7B / 'run_spec.json', RUN_SPEC_7B)

LG_ROWS = []
LG_WRITE_STATS = []
for scenario in SCENARIOS:
    sid = scenario['scenario_id']
    for session in scenario['sessions']:
        conf = {'configurable': {'thread_id': f'7b:{SPLIT}:{sid}:session:{session["session_id"]}'}}
        res = GRAPH.invoke({'phase': 'write', 'scenario_id': sid, 'session': session}, conf)
        LG_WRITE_STATS.append({'scenario_id': sid, **res['write_stats']})
        print('Escritura', sid, session['session_id'], res['write_stats']['ops'])
        for p in [x for x in scenario['probes'] if x['after_session']==session['session_id']]:
            for arm in ARMS_7B:
                cfg = {'configurable': {'thread_id': f'7b:{SPLIT}:{sid}:probe:{p["probe_id"]}:{arm}'}}
                r = GRAPH.invoke({'phase': 'probe', 'scenario_id': sid, 'probe': p, 'arm': arm}, cfg)
                LG_ROWS.append(r['row'])
                print(' ', r['row']['item_id'], arm, r['row']['metrics'].get('correct'),
                      r['row']['metrics'].get('outcome'))

expected = sum(len(s['probes']) for s in SCENARIOS) * len(ARMS_7B)
assert len(LG_ROWS)==expected, f'Faltan resultados: {len(LG_ROWS)} / {expected}'
write_jsonl(RESULTS_7B / 'framework_rows.jsonl', LG_ROWS)
write_json(RESULTS_7B / 'write_stats.json', LG_WRITE_STATS)
print('RESULTADOS 7B:', RESULTS_7B, 'Filas:', len(LG_ROWS))

Escritura m002 s1 {'ADD': 7, 'UPDATE': 0, 'NOOP': 0, 'INVALIDATE': 0}
  m002/p1 memory True answered
  m002/p1 memory_filter True answered
  m002/p8 memory True answered
  m002/p8 memory_filter True answered
Escritura m002 s2 {'ADD': 1, 'UPDATE': 1, 'NOOP': 0, 'INVALIDATE': 0}
  m002/p2 memory True answered
  m002/p2 memory_filter True answered
  m002/p3 memory True answered
  m002/p3 memory_filter True answered
  m002/p4 memory True answered
  m002/p4 memory_filter True answered
  m002/p9 memory True answered
  m002/p9 memory_filter True answered
Escritura m002 s3 {'ADD': 0, 'UPDATE': 1, 'NOOP': 0, 'INVALIDATE': 0}
  m002/p5 memory True answered
  m002/p5 memory_filter True answered
  m002/p6 memory True abstained
  m002/p6 memory_filter True abstained
  m002/p7 memory True answered
  m002/p7 memory_filter True answered
Escritura m003 s1 {'ADD': 7, 'UPDATE': 0, 'NOOP': 0, 'INVALIDATE': 0}
  m003/p1 memory True answered
  m003/p1 memory_filter True answered
  m003/p8 memory True answer

## Comparación pareada con la Etapa 6

Se busca `probes.jsonl` de una corrida **Etapa 6 full/test**, con mismo Qwen, `num_ctx`, corpus e índice. Si no existe, la comparación **no se inventa**: quedan los resultados 7b guardados y se indica cómo copiar la carpeta original de la Etapa 6 a `artifacts/full/results/`.

Cada fila se empareja por `(item_id, arm)`. Se calcula acuerdo en corrección, exactitud en cada brazo, diferencia de tokens de entrada y salida, y diferencia textual (no se exige texto idéntico a temperatura 0). Se deja la trazabilidad de cada sonda en CSV.

In [18]:
from pathlib import Path
import shutil
import json

base = Path("/content/drive/MyDrive/projecto_rag")

origen = (
    base / "artifacts_respaldo_20261101_215934"
    / "full/results/stage6_14e442282fc21efb"
)

destino = (
    base / "artifacts/full/results"
    / "stage6_14e442282fc21efb"
)

# Verificar los resultados originales
assert (origen / "probes.jsonl").is_file(), "Falta probes.jsonl"
assert (origen / "run_spec.json").is_file(), "Falta run_spec.json"

# Copiar sin sobrescribir archivos existentes
destino.mkdir(parents=True, exist_ok=True)

for archivo in origen.rglob("*"):
    if archivo.is_file():
        salida = destino / archivo.relative_to(origen)
        salida.parent.mkdir(parents=True, exist_ok=True)
        if not salida.exists():
            shutil.copy2(archivo, salida)

# Comprobar compatibilidad
spec = json.loads((destino / "run_spec.json").read_text())

print("Baseline recuperada correctamente.")
print("Modelo:", spec["model"])
print("Split:", spec["split"])
print("Índice:", spec["stage3_index"])
print("Prompts compatibles:",
      spec.get("prompts_sha256") == RUN_SPEC_7B["prompts_sha256"])
print("Datos de sondas:", (destino / "probes.jsonl").is_file())

Baseline recuperada correctamente.
Modelo: qwen3:8b
Split: test
Índice: stage3_36a95c9c1551289f
Prompts compatibles: True
Datos de sondas: True


In [19]:
import csv


def extract_tokens(row):
    info = row.get('llm') or {}
    prompt = info.get('prompt_tokens')
    output = info.get('output_tokens')
    # Registro del wrapper del curso (algunos campos pueden venir como None).
    if prompt is None:
        prompt = info.get('prompt_eval_count')
    if output is None:
        output = info.get('eval_count')
    return prompt, output


def find_stage6_baseline():
    candidates = sorted((ART_FULL / 'results').glob('stage6_*/probes.jsonl'))
    good = []
    for path in candidates:
        try:
            spec = json.loads((path.parent / 'run_spec.json').read_text(encoding='utf-8'))
            cfg = spec.get('config', {})
            if (spec.get('split') == SPLIT and spec.get('model') == MODEL and
                cfg.get('num_ctx') == CFG.num_ctx and cfg.get('passage_budget') == CFG.passage_budget and
                cfg.get('memory_budget') == CFG.memory_budget and
                spec.get('stage3_index') == INDEX_DIR.name and
                spec.get('corpus_hash') == FROZEN['documents_sha256'] and
                spec.get('prompts_sha256') == RUN_SPEC_7B['prompts_sha256'] and
                spec.get('embedder',{}).get('model') == EMBEDDER_ID.get('model')):
                good.append(path)
        except (OSError, KeyError, ValueError):
            pass
    return good

baseline_candidates = find_stage6_baseline()
if len(baseline_candidates) > 1:
    print('Varias corridas compatibles. Usando la última ordenada por ruta:', [str(p) for p in baseline_candidates])
if not baseline_candidates:
    print('FALTA BASELINE: no encontré el resultado de Etapa 6 con las mismas condiciones.')
    print('Copiá desde el Drive original la carpeta stage6_* a artifacts/full/results y volvé a correr ESTA celda.')
else:
    src = baseline_candidates[-1]
    baseline_rows = [r for r in read_jsonl(src) if r.get('arm') in ARMS_7B]
    keys = {(r['item_id'],r['arm']) for r in LG_ROWS}
    manual = {(r['item_id'],r['arm']): r for r in baseline_rows if (r['item_id'],r['arm']) in keys}
    if len(manual) != len(keys):
        print(f'BASELINE INCOMPLETA: {len(manual)}/{len(keys)} comparaciones; no se reporta un resultado global.')
    else:
        def avg(vals): return sum(vals)/len(vals) if vals else None
        table=[]
        for ours in LG_ROWS:
            ref = manual[(ours['item_id'],ours['arm'])]
            op,oo=extract_tokens(ours)
            rp,ro=extract_tokens(ref)
            table.append({
                'item_id':ours['item_id'], 'arm':ours['arm'],
                'manual_correct':ref['metrics']['correct'], 'langgraph_correct':ours['metrics']['correct'],
                'manual_answer':(ref.get('output') or {}).get('answer'),
                'langgraph_answer':(ours.get('output') or {}).get('answer'),
                'same_answer_text':(ref.get('output') or {}).get('answer')==(ours.get('output') or {}).get('answer'),
                'manual_prompt_tokens':rp, 'langgraph_prompt_tokens':op,
                'manual_output_tokens':ro, 'langgraph_output_tokens':oo,
                'delta_prompt_tokens':(op-rp) if op is not None and rp is not None else None,
                'delta_output_tokens':(oo-ro) if oo is not None and ro is not None else None,
                'manual_memory_tokens':ref.get('memory_tokens'),
                'langgraph_memory_tokens':ours.get('memory_tokens'),
                'manual_passage_tokens':ref.get('passage_tokens'),
                'langgraph_passage_tokens':ours.get('passage_tokens'),
                'same_delivered':ref.get('delivered')==ours.get('delivered'),
                'same_memory_ids':ref.get('memory_ids')==ours.get('memory_ids'),
            })
        with (RESULTS_7B/'paired_comparison.csv').open('w',newline='',encoding='utf-8') as fh:
            writer=csv.DictWriter(fh,fieldnames=list(table[0]));writer.writeheader();writer.writerows(table)
        summaries=[]
        for arm in ARMS_7B:
            vals=[r for r in table if r['arm']==arm]
            aok=sum(r['manual_correct'] is True for r in vals)
            bok=sum(r['langgraph_correct'] is True for r in vals)
            diff_p=[r['delta_prompt_tokens'] for r in vals if r['delta_prompt_tokens'] is not None]
            diff_o=[r['delta_output_tokens'] for r in vals if r['delta_output_tokens'] is not None]
            summaries.append({'arm':arm,'n':len(vals),'manual_correct':aok,'langgraph_correct':bok,
                              'agreement_correct':sum(r['manual_correct']==r['langgraph_correct'] for r in vals),
                              'same_answer_text':sum(r['same_answer_text'] for r in vals),
                              'same_delivered':sum(r['same_delivered'] for r in vals),
                              'same_memory_ids':sum(r['same_memory_ids'] for r in vals),
                              'mean_delta_prompt_tokens':avg(diff_p),'mean_delta_output_tokens':avg(diff_o)})
        write_json(RESULTS_7B/'paired_summary.json',summaries)
        print('Resumen por brazo (comparación pareada):')
        for line in summaries:print(line)
        print('Comparaciones guardadas en:',RESULTS_7B/'paired_comparison.csv')

Resumen por brazo (comparación pareada):
{'arm': 'memory', 'n': 18, 'manual_correct': 18, 'langgraph_correct': 18, 'agreement_correct': 18, 'same_answer_text': 14, 'same_delivered': 18, 'same_memory_ids': 18, 'mean_delta_prompt_tokens': 0.0, 'mean_delta_output_tokens': 0.4444444444444444}
{'arm': 'memory_filter', 'n': 18, 'manual_correct': 18, 'langgraph_correct': 18, 'agreement_correct': 18, 'same_answer_text': 16, 'same_delivered': 18, 'same_memory_ids': 18, 'mean_delta_prompt_tokens': 0.0, 'mean_delta_output_tokens': 2.5555555555555554}
Comparaciones guardadas en: /content/projecto_rag/artifacts/full/results/stage7b_langgraph_b326f7f25e64154f/paired_comparison.csv


## Exportar artefactos y revisar decisiones del framework

1. Guardar `run_spec.json`, `framework_rows.jsonl`, `write_stats.json` y (si la baseline está disponible) `paired_comparison.csv` y `paired_summary.json`.
2. Conservar este `.ipynb` ejecutado **con todas las salidas** en el ZIP final; subir al repo una versión limpia si `tools/check_repo.py` rechaza outputs.
3. Documentar en `decision_log.md` el alcance: *LangGraph realiza el grafo y la persistencia, no decide qué memorias extraer, cuándo resumir, ni los límites de tokens*.
4. En el informe, comparar los números realmente obtenidos, no los 18/18 anteriores como si fueran resultados de LangGraph.

In [20]:
from pathlib import Path
import shutil, zipfile

zip_path = RESULTS_7B.parent / (RESULTS_7B.name + '.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RESULTS_7B.rglob('*')):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(RESULTS_7B.parent)))
print('ZIP resultados 7b:',zip_path)

# Respaldo cuando artifacts/ no apunta a Google Drive
if environment.on_colab() and not environment.ARTIFACTS.is_symlink():
    to_drive = DRIVE_ART / PROFILE / 'results' / RESULTS_7B.name
    to_drive.mkdir(parents=True,exist_ok=True)
    for path in RESULTS_7B.iterdir():
        if path.is_file():shutil.copy2(path,to_drive/path.name)
    print('Respaldo copiado a:',to_drive)

ZIP resultados 7b: /content/projecto_rag/artifacts/full/results/stage7b_langgraph_b326f7f25e64154f.zip
